# 02 · From-Scratch DPO: HH-RLHF and UltraFeedback
**PA2 Task 4 (Sec. 10).** Both runs start from the SFT **best** checkpoint and use the same DPO code. Expected runtime on T4 x2: ~1-1.5 h per run.

**Before running:** *Add Input → Your Work → Notebooks →* the saved version of notebook 01 (it contains `outputs/sft/best`).

How one DPO update works here (PA2 Sec. 2.4.3, RA2 Sec. 4):
1. Same ChatML prompt for chosen `y_w` and rejected `y_l`; prompt tokens masked.
2. **Phase A** – a frozen copy of the SFT model (π_ref) scores every `log π_ref(y|x)` once with `no_grad`, on both GPUs; then it is freed. (π_ref never changes, so these are constants; RA2 App. B's `dpo_loss` also receives `ref_logps` as an input.)
3. **Phase B** – the policy π_θ scores chosen+rejected in one forward pass; loss `-log σ(β[(log π_θ(y_w)-log π_θ(y_l)) - (log π_ref(y_w)-log π_ref(y_l))])` (PA2 Eq. 5 = RA2 Eq. 7); gradients flow only into π_θ.
4. 1 pair per GPU × 16 accumulation × 2 GPUs = 32 pairs per step, β = 0.1, lr 1e-6 constant, RMSprop (RA2 App. B), clip 1.0.
5. Validation every 50 steps: DPO loss, reward accuracy (r_w > r_l), reward margin (PA2 Eq. 7-10). `best/` = highest val reward accuracy.

## Step 0 · Copy the code into Kaggle's working folder  (PA2 Task 1, Sec. 7 + Sec. 5.4)
Notebook settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified Kaggle account).

The code comes from your uploaded Kaggle Dataset (the `small-model-alignment.zip`). Attach it with **+ Add Input → Datasets → Your Datasets**. This cell finds it automatically, so there are no paths to edit.

In [ ]:
# =============================================================================
# Step 0: copy the code from the attached Kaggle Dataset into /kaggle/working
#   /kaggle/working is the ONLY folder Kaggle keeps when you "Save Version" (PA2 Sec. 5.5)
# =============================================================================
import os, shutil, glob

REPO_DIR = "/kaggle/working/small-model-alignment"

def find_code_dataset():
    # Locate the uploaded code dataset anywhere under /kaggle/input.
    # Attached notebook OUTPUTS also contain a copy of the repo, but those copies
    # have an outputs/ folder; the clean code dataset does not, so prefer it.
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)]
    clean = [h for h in hits if not os.path.isdir(os.path.join(h, "outputs"))]
    if not (clean or hits):
        raise FileNotFoundError("Code dataset not found: + Add Input -> Datasets -> Your Datasets -> your code dataset")
    return (clean or hits)[0]

if not os.path.exists(os.path.join(REPO_DIR, "submission_adapter.py")):
    src = find_code_dataset()
    print("copying code from", src)
    shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
    !chmod -R u+w {REPO_DIR}   # /kaggle/input is read-only; make the copy writable
%cd {REPO_DIR}

# Memory/perf environment for the T4s, inherited by every `!python` / `!torchrun` below:
#   expandable_segments -> less CUDA fragmentation with dynamic (per-batch) padding
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def sh(cmd):
    # Run a shell command, stream its output here, and STOP the notebook if it fails
    # (so a crashed torchrun does not silently continue during Save & Run All).
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"command failed: {cmd[:150]}")
!ls

## Step 1 · Install dependencies without touching PyTorch  (PA2 Sec. 5.2)
Kaggle's preinstalled PyTorch already matches the T4 CUDA runtime, so we never reinstall it. `torchao` is removed because old Kaggle builds break PEFT/LoRA loading.

In [ ]:
# =============================================================================
# Step 1: install the pinned libraries (exactly the manual's safe pattern)
# =============================================================================
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

## Step 2 · Verify the two T4 GPUs and print versions  (PA2 Task 1 items 3-4)
Expect two *Tesla T4* rows with ~15 GiB each. T4 = Turing (sm_75): fast **fp16** tensor cores, **no bf16**, no FlashAttention-2 → the code uses fp16 autocast + PyTorch SDPA attention.

In [ ]:
# =============================================================================
# Step 2: GPU + package check
# =============================================================================
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs visible:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Set Accelerator = GPU T4 x2 in the notebook settings"
NPROC = torch.cuda.device_count()

## Step 3 · Locate the SFT checkpoint from the attached notebook-01 output

In [ ]:
# =============================================================================
# Helper: find a checkpoint inside attached Kaggle inputs (PA2 Sec. 5.4 items 5-6)
#   A saved notebook version attached as input appears under /kaggle/input/...
# =============================================================================
def find_ckpt(rel):
    """rel like 'outputs/sft/best' -> absolute folder that contains config.json"""
    local = os.path.join(REPO_DIR, rel)
    if os.path.exists(os.path.join(local, "config.json")):
        return local
    hits = sorted(glob.glob(f"/kaggle/input/**/{rel}/config.json", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{rel} not found - did you attach the previous notebook's output (Add Input)?")
    return os.path.dirname(hits[0])

SFT_BEST = find_ckpt("outputs/sft/best")
print("SFT best checkpoint:", SFT_BEST)
!cat {SFT_BEST}/checkpoint_info.json | head -n 5

## Step 4 · Prepare preference data  (PA2 Sec. 6.3)
Single-turn pairs only (multi-turn HH dialogues are filtered out, Sec. 6.3), 15,000 train / 1,000 val for each dataset, seed 42.

In [ ]:
# =============================================================================
# Step 4: HH-RLHF + UltraFeedback pairs
# =============================================================================
!python scripts/prepare_data.py --only hh uf alpaca   # alpaca: needed by the Step 8 sanity check
!wc -l data/processed/*.jsonl

## Step 5 · DPO smoke test (~3 min)
At step 0 the policy equals the reference, so the log must show `val_loss=0.6931` (= log 2), `val_reward_acc=0.0`, `val_reward_margin=0.0` (PA2 Sec. 10.5). If you see that, the masking and log-prob code are consistent.

In [ ]:
# =============================================================================
# Step 5: tiny DPO run
# =============================================================================
sh(f"torchrun --nproc_per_node {NPROC} scripts/train_dpo.py --sft_path {SFT_BEST} --data hh "
   "--output_dir outputs/dpo_smoke --max_train_examples 128 --max_val_examples 32 --eval_every 2 --log_every 1")
!rm -rf outputs/dpo_smoke

## Step 6 · Full DPO runs  (PA2 Sec. 10.2-10.4)
`RUNS` controls which tracks run. Each run writes `outputs/dpo_<data>/{metrics.jsonl, best/, final/, ref_logps_*.pt}`; a finished run is skipped, and the cached reference log-probs make a restart skip Phase A.

In [ ]:
# =============================================================================
# Step 6: HH-RLHF DPO, then UltraFeedback DPO
# =============================================================================
RUNS = ["hh", "uf"]
for data in RUNS:
    out = f"outputs/dpo_{data}"
    if os.path.exists(f"{out}/final/config.json"):
        print("skip (already finished):", out); continue
    sh(f"torchrun --nproc_per_node {NPROC} scripts/train_dpo.py --sft_path {SFT_BEST} --data {data} "
       f"--output_dir {out} --beta 0.1 --lr 1e-6 --optimizer rmsprop --per_device_batch_size 1 "
       "--effective_batch_size 32 --max_length 512 --max_prompt_length 256 --epochs 1 --eval_every 50")

## Step 7 · Plots: DPO loss, reward accuracy, reward margin  (PA2 Sec. 14.4)

In [ ]:
# =============================================================================
# Step 7: curves for both runs
# =============================================================================
from IPython.display import Image, display
for data in ["hh", "uf"]:
    if os.path.exists(f"outputs/dpo_{data}/metrics.jsonl"):
        !python scripts/plot_metrics.py --run outputs/dpo_{data} --kind dpo
        display(Image(f"outputs/dpo_{data}/dpo_{data}_curves.png"))
        !cat outputs/dpo_{data}/best/checkpoint_info.json | head -n 12

In [ ]:
# =============================================================================
# Step 8: sanity-check each DPO checkpoint on 3 prompts
# =============================================================================
!python scripts/generate.py --model_path outputs/dpo_hh/best --generator dpo_hh --limit 3 --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output outputs/generations/_sanity_hh.json
!python scripts/generate.py --model_path outputs/dpo_uf/best --generator dpo_uf --limit 3 --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/_sanity_uf.json
!rm -f outputs/dpo_*/ref_logps_*.pt   # caches no longer needed
!zip -q -r /kaggle/working/dpo_run_outputs.zip outputs -x "*.safetensors"
!du -sh outputs/*

## Final step · Preserve the outputs  (PA2 Sec. 5.5)
1. Click **Save Version → Save & Run All (Commit)** (or, if you ran interactively, *Quick Save* keeps the current outputs).
2. After the version finishes, its `/kaggle/working` folder becomes an **Output** you can attach to the next notebook with **Add Input → Your Work → Notebooks**.